In [36]:
from pathlib import Path
import pandas as pd

# Diretório com os arquivos CSV
csv_dir = Path("CSV")
files = sorted(csv_dir.glob("*.csv"))

# Lê todos os CSVs e constrói o DataFrame
linhas = []
nomes_amostras = []

for f in files:
    nomes_amostras.append(f.name)
    df_csv = pd.read_csv(
        f,
        sep=';',
        decimal=',',
        header=0,
        names=["Comprimento de onda", "Valor"]
    )
    row = df_csv.set_index("Comprimento de onda")["Valor"]
    linhas.append(row)

# Cria o DataFrame com amostras como índice
if linhas:
    df = pd.DataFrame(linhas)
    df.index = nomes_amostras
    df.index.name = "amostra"
else:
    df = pd.DataFrame(columns=[nomes_amostras], index=nomes_amostras)

# Lê o arquivo com as colunas adicionais
df_mpc = pd.read_excel("dataset_pca.xlsx", index_col=0)
df_mpc.index.name = "amostra"

# Mescla os dois DataFrames pelo índice
df = df.merge(df_mpc, left_index=True, right_index=True, how="inner")
df

,3999.6401,3998.673536,3997.706973,3996.740409,3995.773845,3994.807281,3993.840718,3992.874154,3991.90759,3990.941027,...,405.956182,404.989619,404.023055,403.056491,402.089927,401.123364,400.1568,ponto de fulgor,massa específica,enxofre
amostra,,,,,,,,,,,,,,,,,,,,,
35713.csv,0.011040,0.004737,-0.005739,-0.017375,-0.026226,-0.028694,-0.024322,-0.016353,-0.008132,-0.001908,...,-0.005950,-0.005682,-0.005390,-0.005084,-0.004801,-0.004574,-0.004421,32.9,844.9,160.0
35718.csv,-0.014060,-0.014143,-0.013894,-0.014723,-0.015764,-0.015720,-0.015115,-0.015088,-0.015042,-0.013952,...,-0.010162,-0.009631,-0.009098,-0.008565,-0.008069,-0.007658,-0.007353,35.0,858.1,340.0
35721.csv,-0.016433,-0.017012,-0.016996,-0.018368,-0.020467,-0.021432,-0.021232,-0.021132,-0.020891,-0.019497,...,-0.005919,-0.005832,-0.005729,-0.005612,-0.005506,-0.005428,-0.005384,47.1,840.4,7.3
35724.csv,-0.006937,-0.009066,-0.011083,-0.011950,-0.011659,-0.011398,-0.012215,-0.013108,-0.012081,-0.009361,...,-0.009732,-0.009370,-0.008996,-0.008611,-0.008246,-0.007946,-0.007732,48.0,858.8,360.0
35730.csv,-0.013479,-0.014290,-0.014110,-0.015149,-0.016833,-0.017377,-0.016846,-0.016471,-0.015954,-0.014469,...,-0.004714,-0.004456,-0.004173,-0.003868,-0.003582,-0.003365,-0.003232,52.0,837.8,150.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39008.csv,0.067620,0.036828,0.015539,0.003128,0.000553,0.006785,0.017271,0.026757,0.033228,0.035432,...,-0.036533,-0.036355,-0.036168,-0.035921,-0.035634,-0.035393,-0.035248,49.2,839.8,0.0
39011.csv,0.026867,0.026390,0.024355,0.020439,0.017657,0.017833,0.019240,0.019558,0.019180,0.019203,...,-0.030163,-0.030427,-0.030642,-0.030767,-0.030837,-0.030938,-0.031102,47.0,850.7,170.0
39018.csv,0.027644,0.024312,0.020926,0.017318,0.015790,0.017284,0.019473,0.019953,0.019517,0.019595,...,-0.024676,-0.025411,-0.026052,-0.026574,-0.027030,-0.027502,-0.028004,49.0,857.8,320.0


In [ ]:
import plotly.express as px

# Cria um DataFrame limpo para o gráfico, sem colunas duplicadas
plot_df = pc_scores.copy()
plot_df = plot_df.loc[:, ~plot_df.columns.duplicated()].copy()

# Remove qualquer coluna 'amostra' antiga para evitar duplicidade
if 'amostra' in plot_df.columns:
    plot_df = plot_df.drop(columns=['amostra'])

# Adiciona a coluna de amostras de forma única
plot_df.insert(0, 'amostra', df.index.tolist())

# Adiciona as variáveis alvo, se existirem
for col in ['ponto de fulgor', 'massa específica', 'enxofre']:
    if col in df.columns:
        plot_df[col] = df[col].values

# Mantém apenas os PCs como eixos do gráfico
pc_columns = [c for c in plot_df.columns if c.startswith('PC')]
plot_df = plot_df[['amostra'] + pc_columns + [c for c in ['ponto de fulgor', 'massa específica', 'enxofre'] if c in plot_df.columns]]

# Gráfico completo com todos os PCs
fig = px.parallel_coordinates(
    plot_df,
    dimensions=pc_columns,
    color='enxofre' if 'enxofre' in plot_df.columns else None,
    labels={c: c for c in pc_columns},
    title='PCA - Todos os PCs para todas as amostras'
)

fig.update_layout(width=1400, height=800)
fig.show()

# Opcional: salva em HTML
fig.write_html('pca_all_pcs.html')

In [ ]:
import plotly.express as px

# Preparação do DataFrame para o gráfico de dispersão
plot_df = pc_scores.copy()
plot_df = plot_df.loc[:, ~plot_df.columns.duplicated()].copy()

if 'amostra' in plot_df.columns:
    plot_df = plot_df.drop(columns=['amostra'])

plot_df.insert(0, 'amostra', df.index.tolist())

for col in ['ponto de fulgor', 'massa específica', 'enxofre']:
    if col in df.columns:
        plot_df[col] = df[col].values

# Escolhe os dois primeiros componentes para o gráfico de pontos
pc_columns = [f'PC{i}' for i in range(1, 3)]
plot_df = plot_df[['amostra'] + pc_columns + [c for c in ['ponto de fulgor', 'massa específica', 'enxofre'] if c in plot_df.columns]]

# Gráfico de PCA em pontos
fig = px.scatter(
    plot_df,
    x='PC1',
    y='PC2',
    text='amostra',
    color='enxofre' if 'enxofre' in plot_df.columns else None,
    hover_data=['amostra', 'ponto de fulgor', 'massa específica', 'enxofre'],
    color_continuous_scale='Viridis',
    title='PCA - Gráfico de dispersão das amostras'
)

fig.update_traces(textposition='top center', marker=dict(size=10))
fig.update_layout(width=1200, height=800)
fig.show()

# Salva o gráfico em HTML
fig.write_html('pca_scatter.html')